In [1]:
from crewai.tools import BaseTool

class RAGTool(BaseTool):
    name: str = "RAGTool"
    description: str = "Retrieve from ChromaDB first, fallback to Gemini web search."

    def __init__(self, retriever, web_tool):
        self.retriever = retriever
        self.web_tool = web_tool

    def _run(self, query: str) -> str:
        # 1. Lấy docs từ Chroma
        docs = self.retriever.get_relevant_documents(query)
        if docs:
            # nối nội dung doc để LLM trả lời
            combined_text = "\n".join([doc.page_content for doc in docs])
            return combined_text
        else:
            # fallback ra websearch
            return self.web_tool._run(query)


In [2]:
from crewai.tools import BaseTool
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os

load_dotenv()

class GeminiGoogleSearchTool(BaseTool):
    name:str = "GeminiGoogleSearch"
    description:str = "Search the web using Google's native Gemini Search grounding."

    def _run(self, query: str) -> str:
        client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

        grounding_tool = types.Tool(google_search=types.GoogleSearch())
        config = types.GenerateContentConfig(tools=[grounding_tool])

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=query,
            config=config,
        )
        return response.text

gemini_search_tool = GeminiGoogleSearchTool()


In [7]:
from langchain_openai import ChatOpenAI

# Kiểm tra OPENAI_API_KEY
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Vui lòng cung cấp OPENAI_API_KEY trong file .env")
# Initialize the language model using OpenAI
llm = ChatOpenAI(
    model="gpt-4o-mini",  # Hoặc "gpt-4" nếu bạn cần model mạnh hơn
    temperature=0.7,
    verbose=True
)

ValueError: Vui lòng cung cấp OPENAI_API_KEY trong file .env

In [ ]:
# import google.generativeai as genai
# genai.configure(api_key=os.getenv('GOOGLE_API_KEY'))
# llm = genai.GenerativeModel(model_name="gemini-2.5-flash")

In [ ]:
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document


In [ ]:
# 1. Tạo embeddings (dùng khi query)
embeddings = HuggingFaceEmbeddings(model_name="AITeamVN/Vietnamese_Embedding", model_kwargs={"device": "cpu"})

# 2. Kết nối ChromaDB
chroma_db = Chroma(
    persist_directory="./chroma_db",  # thư mục bạn đã lưu DB
    embedding_function=embeddings
)

# 3. Tạo retriever
retriever = chroma_db.as_retriever(
    search_kwargs={"k": 5}  # số lượng doc top-k
)


In [ ]:
from crewai.tools import BaseTool
from typing import Any

class RAGTool(BaseTool):
    name: str = "RAGTool"
    description: str = "Retrieve from ChromaDB first, fallback to Gemini web search."

    retriever: Any = None   
    web_tool: Any = None    

    def _run(self, query: str) -> str:
        # 1. Thử lấy docs từ retriever
        docs = []
        if self.retriever:
            docs = self.retriever.get_relevant_documents(query)

        if docs and len(docs) > 0:
            combined_text = "\n".join([doc.page_content for doc in docs])
            return f"Retrieved from ChromaDB:\n{combined_text}"
        else:
            return f"Fallback to Web:\n{self.web_tool._run(query)}"


In [ ]:
# Giả sử bạn đã có:
# - retriever: 1 object trả về danh sách Document hoặc (Document, score)
# - gemini_search_tool: instance từ trước

rag_tool = RAGTool(
    retriever=retriever,          # ví dụ: retriever = my_vector_store.as_retriever()
    web_tool=gemini_search_tool,     # dùng tool Gemini
)

# Gắn tool này vào agent:
from crewai import Agent, Task, Crew

rag_agent = Agent(
    role="RAG Specialist",
    goal="Answer user questions using internal docs first, else web",
    backstory="Combine vector retrieval and websearch",
    llm=llm,
    tools=[rag_tool],
    verbose=True
)

rag_task = Task(name="RAG task", description="Answer {topic}", agent=rag_agent, expected_output="Final answer")
crew = Crew(agents=[rag_agent], tasks=[rag_task], verbose=True, process="sequential")

def run(query):
    return crew.kickoff(inputs={"topic": query})


In [ ]:
# Query nội bộ
print(run("VỊ TRÍ VIỆC LÀM CỦA SINH VIÊN TỐT NGHIỆP NGANH Kinh tế công nghiệp LA GI "))

# Query ngoài web
print(run("Who won the Euro 2024?"))


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 593d2f21-42c3-4d82-9fca-bc67d54e7ca2                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Specialist                                                                                          │
│                                                                                                                 │
│  Task: Answer VỊ TRÍ VIỆC LÀM CỦA SINH VIÊN TỐT NGHIỆP NGANH Kinh tế công nghiệp LA GI                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── LLM Error ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ LLM Call Failed                                                                                             │
│  Error: litellm.BadRequestError: GetLLMProvider Exception - argument of type 'URL' is not iterable              │
│                                                                                                                 │
│  original model: <openai.OpenAI object at 0x7a60704c3a90>                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Task Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: RAG task                                                                                                 │
│  Agent: RAG Specialist                                                                                          │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 593d2f21-42c3-4d82-9fca-bc67d54e7ca2                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output:                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

BadRequestError: litellm.BadRequestError: GetLLMProvider Exception - argument of type 'URL' is not iterable

original model: <openai.OpenAI object at 0x7a60704c3a90>